# Phase 7 · The story: where every number in the README comes from

The README and the post drafts are **generated**: `story/*.md.tmpl` holds the words and `{placeholders}`;
`src/story_numbers.py` computes every figure from `data/processed/`; `pipeline/build_story.py` fills them in.
`tests/test_story.py` fails if anyone types a number into a template.

This notebook shows the two new pieces of analysis behind beat 1 ("The patient"), then the full list of
story numbers.

In [1]:
import sys
from pathlib import Path
import pandas as pd
import plotly.io as pio
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path[:0] = [str(ROOT / "pipeline"), str(ROOT / "src")]
import charts, story
from story_numbers import story_numbers
pio.renderers.default = "plotly_mimetype"
pd.set_option("display.max_rows", 200, "display.width", 180, "display.max_colwidth", 120)
P = ROOT / "data" / "processed"
halima = story.halima()["state"]

## 1. How far is the nearest working dialysis unit?
- **Destinations:** the dialysis units the register names as working, located via GRID3
  (`data/lookups/dialysis_unit_locations.csv`; two proxies and one unverified match are flagged there).
- **Origins:** each LGA's approximate centre (the median position of its GRID3 facilities), weighted by
  GRID3/WorldPop LGA population.
- **Distance:** straight line. An upper bound: the register is a floor, and roads are longer.

In [2]:
units = pd.read_csv(ROOT / "data/lookups/dialysis_unit_locations.csv")
print(units.match.value_counts().to_dict())
units[units.match != "name match"][["state", "facility_name", "grid3_name", "match", "note"]]

{'name match': 24, 'proxy': 2, 'unverified': 1}


,state,facility_name,grid3_name,match,note
1,Bauchi,Abubakar Tafawa Balewa University Teaching Hospital,Bauchi Hospital Specialist,proxy,"PROXY: ATBUTH is not in GRID3 (eHealth Africa lists it, without coordinates); location of Bauchi State Specialist Ho..."
6,Ekiti,"Federal Teaching Hospital, Ekiti (source says Ado-Ekiti; likely Ido-Ekiti, unverified)",Ido Ekiti Federal Teaching Hospital,unverified,UNVERIFIED: register is unsure whether this is Ido-Ekiti; GRID3 Federal Teaching Hospital Ido-Ekiti used
9,Jigawa,"Fatima Abubakar Dialysis Centre, Hadejia",Hadejia General Hospital,proxy,PROXY: not in GRID3; Hadejia General Hospital in the same town used


In [3]:
lga = pd.read_csv(P / "lga_dialysis_distance.csv")
print(lga.match.value_counts().to_dict(), "(LGA name joins between the population file and GRID3)")
lga[lga.state == halima].sort_values("km_straight_line")[["lga", "population", "nearest_unit", "km_straight_line"]]

{'exact': 743, 'closest spelling': 28, 'no match': 3} (LGA name joins between the population file and GRID3)


,lga,population,nearest_unit,km_straight_line
91,Bauchi,1.201327e+06,Abubakar Tafawa Balewa University Teaching Hospital,5.0
109,Zaki,4.300094e+05,"Fatima Abubakar Dialysis Centre, Hadejia",35.0
97,Ganjuwa,7.684828e+05,Abubakar Tafawa Balewa University Teaching Hospital,49.0
95,Dass,1.078709e+05,Abubakar Tafawa Balewa University Teaching Hospital,52.0
90,Alkaleri,5.301815e+05,Abubakar Tafawa Balewa University Teaching Hospital,64.0
102,Kirfi,2.739441e+05,Abubakar Tafawa Balewa University Teaching Hospital,65.0
99,Itas/Gadau,3.664649e+05,"Fatima Abubakar Dialysis Centre, Hadejia",66.0
96,Gamawa,4.642538e+05,"Fatima Abubakar Dialysis Centre, Hadejia",71.0
106,Tafawa-Balewa,3.363406e+05,Abubakar Tafawa Balewa University Teaching Hospital,72.0
107,Toro,7.988005e+05,Abubakar Tafawa Balewa University Teaching Hospital,79.0


In [4]:
dist = pd.read_csv(P / "state_dialysis_distance.csv")
print(story.distance_sentence(dist, halima))
charts.distance_to_dialysis(dist, halima).show()

Half of Bauchi's people live more than 72 km (straight line) from a dialysis unit known to work, and 21% more than 100 km.


## 2. Income yardsticks
NBS publishes no average household income in naira, so the story uses the minimum wage and poverty rates.

In [5]:
inc = pd.read_csv(ROOT / "data/manual/income_benchmarks.csv")
inc[["measure", "geography", "value", "unit", "reference_year", "source_date"]]

,measure,geography,value,unit,reference_year,source_date
0,national_minimum_wage,Nigeria,7.000000e+04,NGN per worker per month,2024,2026-10-16
1,national_minimum_wage,Nigeria,7.000000e+04,NGN per worker per month,2024,2024-07
2,total_household_consumption_expenditure,Nigeria,4.020739e+13,"NGN per year (all households, aggregate)",2019,2020-05
3,total_household_consumption_expenditure,North East (excl. Borno),2.469980e+12,"NGN per year (all households, aggregate)",2019,2020-05
4,total_household_consumption_expenditure,Bauchi,7.788002e+11,"NGN per year (all households, aggregate)",2019,2020-05
5,mean_consumption_per_capita_poorest_quintile,Nigeria,6.569000e+04,NGN per person per year,2019,2020-05
6,mean_consumption_per_capita_quintile_2,Nigeria,1.087300e+05,NGN per person per year,2019,2020-05
7,mean_consumption_per_capita_quintile_3,Nigeria,1.542900e+05,NGN per person per year,2019,2020-05
8,mean_consumption_per_capita_quintile_4,Nigeria,2.191950e+05,NGN per person per year,2019,2020-05
9,mean_consumption_per_capita_richest_quintile,Nigeria,4.152540e+05,NGN per person per year,2019,2020-05


## 3. Every number the README quotes

In [6]:
n = story_numbers()
print(len(n), "numbers and sentences, all computed from data")
pd.DataFrame(sorted(n.items()), columns=["key", "value"])

134 numbers and sentences, all computed from data


,key,value
0,app_url,https://nigeria-health-atlas.streamlit.app/
1,band_a,50
2,band_b,100
3,bottom5,"Ekiti, Cross River, Enugu, Kebbi and Ebonyi"
4,built,2026-11-24
5,c_assumed,11
6,c_be_used,87%
7,c_breakeven,"₦907,833"
8,c_capex,₦4.6bn
9,c_driver,maintenance % of equipment per year


In [7]:
import subprocess
r = subprocess.run([sys.executable, "-m", "pytest", "-q", str(ROOT / "tests/test_story.py")], capture_output=True, text=True, cwd=ROOT)
print(r.stdout.strip().splitlines()[-1])

3 passed in 3.36s
